In [ ]:
import urllib.request
import zipfile
import pandas as pd
import pandas as pd
import numpy as np

# Para el preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Para el modelo
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, label_binarize
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, classification_report
)

from scipy.stats import randint
from tempfile import mkdtemp

# MLflow 
import mlflow
import mlflow.sklearn

# URL RAW del archivo
url = "https://raw.githubusercontent.com/Mafegz0/Data/main/df_morosidad.csv.zip"

# Ruta temporal en Databricks
zip_path = "/tmp/df_morosidad.csv.zip"

# Descargar el ZIP
urllib.request.urlretrieve(url, zip_path)

# Descomprimir el ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/tmp")

# Cargar el CSV descomprimido
df = pd.read_csv("/tmp/df_morosidad.csv")

TARGET = "y_categorica"
DROP = ["Llave2", "Nombre_linea", "IDBANNER"]

y = df[TARGET].copy()
X = df.drop([TARGET] + [c for c in DROP if c in df.columns], axis=1).copy()

# Identificar tipo de variables
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

rf_prep = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ],
    remainder="drop"
)


rf = RandomForestClassifier(
    bootstrap=True,
    max_depth=22,
    max_features=0.5,
    min_samples_leaf=5,
    n_estimators=457,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline([
    ("prep", rf_prep),
    ("clf", rf)
])
pipe.set_params(memory=mkdtemp())


experiment = mlflow.set_experiment("/modelo-randomforest")


with mlflow.start_run(experiment_id=experiment.experiment_id):

    pipe.fit(X_train, y_train)
    best_model = pipe

    # Probabilidades para AUC multiclass
    proba_test = best_model.predict_proba(X_test)
    clases = best_model.named_steps["clf"].classes_
    y_test_bin = label_binarize(y_test, classes=clases)

    test_auc = roc_auc_score(
        y_test_bin, proba_test, 
        multi_class="ovr", average="macro"
    )

    y_pred = best_model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)

    params = best_model.named_steps["clf"].get_params()

    # Registrar parámetros
    mlflow.log_param("bootstrap", params["bootstrap"])
    mlflow.log_param("max_depth", params["max_depth"])
    mlflow.log_param("max_features", params["max_features"])
    mlflow.log_param("min_samples_leaf", params["min_samples_leaf"])
    mlflow.log_param("n_estimators", params["n_estimators"])
    mlflow.log_param("class_weight", params["class_weight"])

    # Registrar métricas
    mlflow.log_metric("test_auc_ovr_macro", float(test_auc))
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("precision_macro", float(prec))
    mlflow.log_metric("recall_macro", float(rec))

    # Guardar modelo
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="modelo_rf_morosidad"
    )

    print("\n=== RESULTADOS DEL MODELO FINAL ===")
    print(f"Test AUC (OVR): {test_auc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision macro: {prec:.4f}")
    print(f"Recall macro: {rec:.4f}")

